# Maia Conversational Corpus Builder v3.0 — Notebook 2
## Construção integral, proveniência, auditoria e bloqueio de SFT

**Meta:** 60.000 diálogos (20.000 por idioma) nas cinco categorias. Este notebook não declara o corpus completo se não alcançar cotas e qualidade. **Não inicia treinamento.**

**Importante:** os datasets públicos podem alterar licença, schema e disponibilidade. A aprovação de cada fonte é **manual, explícita e documentada**. Uma licença permissiva declarada no repositório não resolve automaticamente direitos dos dados originais. Nenhuma fonte é aprovada por padrão. O notebook pode gerar exemplos sintéticos determinísticos de lógica/instruções/incerteza, mas eles ficam em quarentena até revisão humana da qualidade e da licença do conteúdo de origem. Não há chamadas pagas a modelos teacher nem alegação de validação LLM-as-judge.

In [ ]:
%pip -q install -U datasets huggingface_hub pandas pyarrow tqdm langdetect

## 1. Configuração e política de liberação

Para iniciar, use `DRY_RUN=True`. Para produção, altere `DRY_RUN=False`, revise `SOURCE_APPROVALS` e `SYNTHETIC_APPROVED`. A aprovação deve ser sustentada por um registro de auditoria, não por um palpite.

In [ ]:
from pathlib import Path
import json, hashlib, random, re, unicodedata, itertools, collections, datetime, os, math
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
SEED=20261120
RNG=random.Random(SEED)
DRY_RUN=True
USE_DRIVE=True
COMMERCIAL_ONLY=True
ALLOW_PARTIAL_EXPORT=True   # exporta somente artefatos de inspeção, nunca declara READY
MAX_PER_SOURCE=400 if DRY_RUN else 200000
TARGETS={'conversation':7000,'comprehension':4000,'logic':4000,'instruction':3000,'uncertainty':2000}
LANGS=('pt','en','es')
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v3_0') if USE_DRIVE else Path('/content/maia_corpus_v3_0')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
OUT=ROOT/('notebook2_dry_run' if DRY_RUN else 'notebook2_full')
OUT.mkdir(parents=True,exist_ok=True)
# Especifique APENAS após auditoria manual: identificador -> justificativa, data, URL da licença e revisão
SOURCE_APPROVALS={}
# Exemplo de formato (NÃO descomentar sem revisão):
# SOURCE_APPROVALS={'oasst2':{'reviewer':'...','reviewed_utc':'...','evidence_url':'...','commercial_ok':True,'upstream_checked':True}}
SYNTHETIC_APPROVED=False # requer auditoria humana de qualidade, diversidade e conteúdo
print('Output:',OUT,'DRY_RUN:',DRY_RUN,'approvals:',list(SOURCE_APPROVALS))

## 2. Registro de fontes e inspeção de metadados

OASST2 é um conjunto de árvores de conversas. Outras fontes têm esquemas heterogêneos. O registro abaixo é um **catálogo de candidatos**, não uma afirmação de compatibilidade jurídica ou disponibilidade. `streaming` evita downloads completos desnecessários.

In [ ]:
SOURCES=[
 {'id':'oasst2','repo':'OpenAssistant/oasst2','split':'train','kind':'oasst_tree','category':'conversation','enabled':True},
 {'id':'ultrachatbr','repo':'recogna-nlp/UltrachatBR','split':'train','kind':'chat','category':'conversation','enabled':True},
 {'id':'aya','repo':'CohereLabs/aya_dataset','split':'train','kind':'pair','category':'conversation','enabled':False},
 {'id':'faquad','repo':'eraldoluis/faquad','split':'train','kind':'qa_context','category':'comprehension','enabled':False},
 {'id':'assin2','repo':'ruanchaves/assin2','split':'train','kind':'nli','category':'comprehension','enabled':False},
 {'id':'no_robots','repo':'HuggingFaceH4/no_robots','split':'train','kind':'chat','category':'instruction','enabled':False},
 {'id':'canarim','repo':'dominguesm/Canarim-Instruct-PTBR-Dataset','split':'train','kind':'pair','category':'instruction','enabled':False},
]
api=HfApi(); manifest=[]
for src in SOURCES:
    d=dict(src)
    try:
        info=api.dataset_info(src['repo']);d['revision']=info.sha;d['card_license']=(info.cardData or {}).get('license') if isinstance(info.cardData,dict) else str(getattr(info.cardData,'license',None));d['metadata_status']='ok'
    except Exception as e:d['metadata_status']='error';d['metadata_error']=str(e)[:400]
    a=SOURCE_APPROVALS.get(src['id'],{})
    d['approved']=bool(a.get('commercial_ok') and a.get('upstream_checked') and a.get('reviewer') and a.get('evidence_url') and a.get('reviewed_utc')) if COMMERCIAL_ONLY else bool(a.get('reviewer') and a.get('evidence_url') and a.get('reviewed_utc'))
    d['approval_evidence']=a
    manifest.append(d)
(OUT/'sources_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf8')
display(pd.DataFrame([{k:x.get(k) for k in ('id','repo','kind','card_license','metadata_status','approved')} for x in manifest]))

## 3. Normalização, validação, identidade de famílias

Amostras sem idioma confiável são postas em quarentena; não se atribui português por omissão. OASST2 precisa reconstruir a árvore completa, não apenas um prefixo de 400 mensagens. O modo completo lê todas as mensagens disponíveis do split antes de montar as cadeias.

In [ ]:
def sha(s):return hashlib.sha256(s.encode('utf8')).hexdigest()
def clean(s):return re.sub(r'\n{4,}','\n\n',unicodedata.normalize('NFC',str(s or '')).replace('\x00','').strip())
def langnorm(x):
    x=str(x or '').lower().replace('_','-')
    return next((l for l in LANGS if x.startswith(l)),None)
def role(x):
    x=str(x or '').lower()
    return 'user' if x in ('user','human','prompter') else ('assistant' if x in ('assistant','gpt','bot') else None)
def validate(ms):
    if not isinstance(ms,list) or not 2<=len(ms)<=14:return 'turn_count'
    if ms[0]['role']!='user' or ms[-1]['role']!='assistant':return 'boundary_roles'
    for i,m in enumerate(ms):
        if m['role'] not in ('user','assistant') or (i and m['role']==ms[i-1]['role']):return 'role_alternation'
        if not 4<=len(m['content'])<=5000:return 'message_length'
    return None
def make(src,raw_id,family,language,ms,category=None,subcategory='imported'):
    language=langnorm(language)
    if language not in LANGS:return None,'unknown_language'
    ms=[{'role':role(m.get('role')),'content':clean(m.get('content'))} for m in ms]
    err=validate(ms)
    if err:return None,err
    canon=json.dumps(ms,ensure_ascii=False,sort_keys=True)
    return {'id':'maia_'+sha(src['id']+'|'+str(raw_id)+'|'+canon)[:24], 'category':category or src['category'],'subcategory':subcategory,'lang':language,'source_id':src['id'],'source':src['repo'],'source_revision':src.get('revision'),'source_record_id':str(raw_id),'family_id':src['id']+'|'+str(family),'messages':ms,'turns':len(ms),'content_sha256':sha(canon),'license_status':'approved' if src.get('approved') else 'quarantine','quality_status':'pending_review'},None
def messages_from_row(row):
    for k in ('messages','conversations','conversation','dialogue','chat','data'):
        v=row.get(k)
        if isinstance(v,list):
            out=[]
            for m in v:
                if not isinstance(m,dict):continue
                r=role(m.get('role',m.get('from',m.get('speaker'))));t=m.get('content',m.get('value',m.get('text')))
                if r and isinstance(t,str):out.append({'role':r,'content':t})
            if out:return out
    return None
def pair_from_row(row):
    p=next((row[k] for k in ('inputs','prompt','instruction','question') if isinstance(row.get(k),str)),None)
    a=next((row[k] for k in ('targets','completion','output','answer','response') if isinstance(row.get(k),str)),None)
    return [{'role':'user','content':p},{'role':'assistant','content':a}] if p and a else None
def qa_from_row(row):
    ctx=next((row[k] for k in ('context','passage','text') if isinstance(row.get(k),str)),None)
    q=next((row[k] for k in ('question','query') if isinstance(row.get(k),str)),None)
    a=row.get('answers',row.get('answer'))
    if isinstance(a,dict):a=a.get('text',[])
    if isinstance(a,list):a=a[0] if a else None
    if not all(isinstance(x,str) and x.strip() for x in (ctx,q,a)):return None
    return [{'role':'user','content':'Texto:\n'+ctx+'\n\nPergunta: '+q},{'role':'assistant','content':a}]
def iter_rows(src):
    ds=load_dataset(src['repo'],split=src['split'],streaming=True,revision=src.get('revision') or None,trust_remote_code=False)
    return itertools.islice(ds,MAX_PER_SOURCE) if DRY_RUN else iter(ds)

## 4. Ingestão real e relatório de erros

Se uma fonte falhar ou seu esquema não for reconhecido, o notebook registra o problema e não preenche a cota artificialmente. OASST2 usa as relações pai–filho; são mantidas apenas cadeias que terminam com resposta do assistente.

In [ ]:
records=[]; ingest_errors=[]; stats=collections.Counter()
for src in manifest:
    if not src['enabled']:continue
    print('Ingesting',src['repo'])
    try:
        if src['kind']=='oasst_tree':
            nodes={}
            for row in iter_rows(src):
                mid=row.get('message_id')
                if mid is not None:nodes[mid]=row
            candidates=[]
            for mid,row in nodes.items():
                if role(row.get('role'))!='assistant':continue
                chain=[];visited=set();cur=row
                while cur and cur.get('message_id') not in visited and len(chain)<14:
                    visited.add(cur['message_id']);r=role(cur.get('role'))
                    if r is None:break
                    chain.append({'role':r,'content':cur.get('text','')})
                    cur=nodes.get(cur.get('parent_id'))
                chain.reverse()
                candidates.append((mid,row.get('message_tree_id',mid),row.get('lang'),chain))
        else:
            candidates=[]
            for i,row in enumerate(iter_rows(src)):
                ms=messages_from_row(row) if src['kind']=='chat' else qa_from_row(row) if src['kind']=='qa_context' else pair_from_row(row) if src['kind']=='pair' else None
                candidates.append((row.get('id',i),row.get('id',i),row.get('lang',row.get('language')),ms))
        for rid,fam,language,ms in candidates:
            if ms is None:stats[(src['id'],'unknown_schema')]+=1;continue
            rec,err=make(src,rid,fam,language,ms)
            if err:stats[(src['id'],err)]+=1
            else:records.append(rec);stats[(src['id'],'normalized')]+=1
    except Exception as e:
        ingest_errors.append({'source':src['id'],'error':str(e)[:1000]});print('ERROR',src['id'],str(e)[:200])
print('normalized',len(records),'errors',len(ingest_errors))
display(pd.DataFrame([{'source':s,'status':k,'count':v} for (s,k),v in stats.items()]))
(OUT/'ingest_errors.json').write_text(json.dumps(ingest_errors,ensure_ascii=False,indent=2),encoding='utf8')

## 5. Complementos sintéticos **verificáveis**

Geração local sem teacher: silogismos simples, instruções de classificação e perguntas de esclarecimento. Cada família lógica usa entidades distintas e rótulo calculado; exemplos de ambiguidade são formulados para pedir dados essenciais. **Não usar esses exemplos como se fossem conversas humanas nem como 12.000 raciocínios independentes.** A aprovação humana e a auditoria de diversidade são obrigatórias. Desabilitado por padrão.

In [ ]:
GENERATE_SYNTHETIC=True
SYNTH_LIMIT_PER_LANG=100 if DRY_RUN else 12000
VOCAB={
 'pt':{'animals':['gato','cachorro','coelho','pássaro','cavalo','peixe'],'colors':['azul','verde','vermelho','amarelo'],'yes':'Sim','no':'Não','ask':'Qual cidade e quantos dias você tem disponíveis?','travel':'Monte um roteiro de viagem para mim.','instruction':'Responda apenas com SIM ou NÃO: a palavra {word} contém a letra {letter}?','logic':'Todos os {plural} são {property}. Este {singular} é um dos {plural}. Ele é {property}?','uncert':'Não tenho informações suficientes para afirmar isso.'},
 'en':{'animals':['cat','dog','rabbit','bird','horse','fish'],'colors':['blue','green','red','yellow'],'yes':'YES','no':'NO','ask':'Which city and how many days do you have?','travel':'Plan a trip for me.','instruction':'Answer only YES or NO: does the word {word} contain the letter {letter}?','logic':'All {plural} are {property}. This {singular} is one of the {plural}. Is it {property}?','uncert':'I do not have enough information to determine that.'},
 'es':{'animals':['gato','perro','conejo','pájaro','caballo','pez'],'colors':['azul','verde','rojo','amarillo'],'yes':'SÍ','no':'NO','ask':'¿A qué ciudad quieres ir y cuántos días tienes?','travel':'Organiza un viaje para mí.','instruction':'Responde solo SÍ o NO: ¿la palabra {word} contiene la letra {letter}?','logic':'Todos los {plural} son {property}. Este {singular} es uno de los {plural}. ¿Es {property}?','uncert':'No tengo información suficiente para determinarlo.'}}
synthetic=[]
if GENERATE_SYNTHETIC:
    for lang,v in VOCAB.items():
        src={'id':'synthetic_rules_v1','repo':'maia:deterministic_templates_v1','revision':'v1','approved':SYNTHETIC_APPROVED,'category':'logic'}
        for i in range(SYNTH_LIMIT_PER_LANG):
            animal=v['animals'][i%len(v['animals'])];color=v['colors'][(i//len(v['animals']))%len(v['colors'])]
            # variações finitas: alta repetição esperada; auditoria de diversidade bloqueia aprovação automática
            typ=i%3
            if typ==0:
                ms=[{'role':'user','content':v['instruction'].format(word=animal,letter='a')},{'role':'assistant','content':v['yes'] if 'a' in animal else v['no']}];cat='instruction';sub='letter_membership'
            elif typ==1:
                # vocabulário não depende de flexão para o rótulo lógico
                premise={'pt':f'Todo objeto da classe A tem cor {color}. O objeto X pertence à classe A. Qual é a cor de X?', 'en':f'Every object in class A has color {color}. Object X belongs to class A. What is the color of X?', 'es':f'Todo objeto de la clase A tiene color {color}. El objeto X pertenece a la clase A. ¿De qué color es X?'}[lang]
                ms=[{'role':'user','content':premise},{'role':'assistant','content':color}];cat='logic';sub='deduction'
            else:
                ms=[{'role':'user','content':v['travel']},{'role':'assistant','content':v['ask']}];cat='uncertainty';sub='missing_information'
            rec,err=make(src,f'{lang}_{i}',f'{lang}_{sub}_{animal}_{color}',lang,ms,cat,sub)
            if rec:synthetic.append(rec)
print('Synthetic candidates:',len(synthetic),'approved?',SYNTHETIC_APPROVED)
records+=synthetic

## 6. Deduplicação, diversidade, aprovação e cotas

O filtro elimina duplicatas exatas normalizadas e reporta concentração de templates. Sem juiz externo ou revisão humana, `quality_status` permanece `pending_review`. Para promover registros à seleção, é necessário um **arquivo de revisão** `quality_approvals.csv` contendo `id,approved,reviewer` (aprovado = true), além da aprovação da fonte. A revisão não é inventada.

In [ ]:
def fingerprint(ms):return sha(re.sub(r'\s+',' ',unicodedata.normalize('NFKC',' '.join(m['role']+':'+m['content'] for m in ms)).casefold()).strip())
seen=set();unique=[];rejected=[]
for r in sorted(records,key=lambda r:sha(str(SEED)+'|'+r['id'])):
    fp=fingerprint(r['messages'])
    if fp in seen:rejected.append({'id':r['id'],'reason':'duplicate_normalized'});continue
    seen.add(fp);unique.append(r)
review_file=ROOT/'quality_approvals.csv'
quality_approved=set()
if review_file.exists():
    q=pd.read_csv(review_file,dtype=str).fillna('')
    if {'id','approved','reviewer'}.issubset(q.columns):
        quality_approved=set(q.loc[(q.approved.str.lower().isin(['true','1','yes'])) & q.reviewer.str.strip().ne(''),'id'])
for r in unique:
    if r['id'] in quality_approved:r['quality_status']='approved'
approved=[r for r in unique if r['license_status']=='approved' and r['quality_status']=='approved']
quarantine=[r for r in unique if r not in approved]
print('unique',len(unique),'approved',len(approved),'quarantine',len(quarantine),'duplicates',len(rejected))
print('quality approvals file:',review_file,'exists?',review_file.exists())
print('Top synthetic template counts:',collections.Counter(r['subcategory'] for r in synthetic))

## 7. Seleção e split por família

As cotas são **por categoria e idioma**. A separação por família evita que uma conversa e suas variações caiam em conjuntos distintos. Se faltarem exemplos, o relatório informa a lacuna sem preenchimento artificial.

In [ ]:
def split(r):
    x=int(sha(str(SEED)+'|'+r['family_id'])[:12],16)/float(16**12)
    return 'train' if x<0.90 else 'validation' if x<0.95 else 'test'
selected=[];counts=collections.Counter()
for r in sorted(approved,key=lambda x:sha(str(SEED)+'|'+x['id'])):
    k=(r['category'],r['lang'])
    if counts[k]<TARGETS.get(r['category'],0):
        selected.append(dict(r,split=split(r)));counts[k]+=1
coverage=[{'category':c,'lang':l,'target':t,'obtained':counts[(c,l)],'missing':t-counts[(c,l)]} for c,t in TARGETS.items() for l in LANGS]
display(pd.DataFrame(coverage))
print('Selected:',len(selected),'missing:',sum(x['missing'] for x in coverage))

## 8. Exportação, auditoria e manifesto SHA-256

`READY_FOR_SFT` só pode ser verdadeiro se: 60.000 selecionados, cotas completas, nenhuma fonte pendente nos registros selecionados, revisão de qualidade aprovada, ausência de erros de ingestão nas fontes habilitadas, ausência de duplicatas e três splits não vazios. **Uma falha não gera arquivo de liberação.**

In [ ]:
def write_jsonl(path,rows):
    with open(path,'w',encoding='utf8') as f:
        for r in rows:f.write(json.dumps(r,ensure_ascii=False)+'\n')
for sp in ('train','validation','test'):write_jsonl(OUT/(sp+'.jsonl'),[r for r in selected if r['split']==sp])
write_jsonl(OUT/'quarantine.jsonl',quarantine)
write_jsonl(OUT/'rejected.jsonl',rejected)
pd.DataFrame(coverage).to_csv(OUT/'coverage.csv',index=False)
status_ready=(len(selected)==60000 and all(x['missing']==0 for x in coverage) and not ingest_errors and all(r['license_status']=='approved' and r['quality_status']=='approved' for r in selected) and len({fingerprint(r['messages']) for r in selected})==len(selected) and all(any(r['split']==sp for r in selected) for sp in ('train','validation','test')) and not DRY_RUN)
report={'seed':SEED,'dry_run':DRY_RUN,'commercial_only':COMMERCIAL_ONLY,'target_total':60000,'selected_total':len(selected),'quarantine_total':len(quarantine),'rejected_total':len(rejected),'missing_total':sum(x['missing'] for x in coverage),'ingest_errors':len(ingest_errors),'status':'READY_FOR_SFT' if status_ready else 'INCOMPLETE_DO_NOT_TRAIN','created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
(OUT/'build_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')
if status_ready:(OUT/'READY_FOR_SFT.json').write_text(json.dumps({'status':'READY_FOR_SFT','report_sha256':hashlib.sha256((OUT/'build_report.json').read_bytes()).hexdigest()},indent=2),encoding='utf8')
else:(OUT/'READY_FOR_SFT.json').unlink(missing_ok=True)
checks=[]
for p in sorted(OUT.iterdir()):
    if p.is_file():checks.append({'file':p.name,'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
pd.DataFrame(checks).to_csv(OUT/'checksums.csv',index=False)
print(json.dumps(report,ensure_ascii=False,indent=2))
display(pd.DataFrame(checks))
assert not status_ready or (OUT/'READY_FOR_SFT.json').exists()
if not status_ready:print('ATENÇÃO: corpus incompleto. NÃO iniciar SFT.')

## 9. Próximas ações para alcançar 60 mil

1. Conferir `sources_manifest.json` e documentar aprovação de proveniência e licenças. Para fontes não comerciais, **não** aprovar no corpus comercial.
2. Ajustar os adaptadores para schemas reais das fontes adicionais, com inspeção de exemplos; habilitar fontes somente após validar schema e licenças.
3. Gerar ou obter diálogos naturais de qualidade para **cada idioma**. Não usar tradução automática sem revisão.
4. Produzir exemplos de lógica, instruções e incerteza com **alta diversidade**, idealmente com teacher e validação independente; o gerador local deste notebook é apenas um teste de infraestrutura.
5. Auditar amostras e criar `quality_approvals.csv` com aprovação humana rastreável. Uma aprovação de centenas de milhares de itens requer processo escalável de curadoria; não marcar como revisados sem evidência.
6. Executar `DRY_RUN=False`, revisar cobertura, contaminação dos benchmarks e distribuição de tokens do tokenizer Maia antes de liberar o SFT.

**Limitação declarada:** este notebook é uma base executável e auditável de construção integral, mas não promete gerar 60 mil diálogos de qualidade a partir de fontes insuficientes ou sem aprovação. A validação semântica, a avaliação de licença upstream e o teste de contaminação por embeddings/benchmarks permanecem etapas a implementar/operar antes de declarar prontidão.